# build chat bot (rag)
## 1 Loading data from PostgreSQL
## 2 Building a search engine
## 3 Build context for model
## 4 Send it to model and print response

In [46]:
import json
import psycopg2
from dotenv import load_dotenv
import os
import requests
from openai import OpenAI

In [47]:
load_dotenv()

conn = psycopg2.connect(
    host = os.getenv('PGHOST'),
    port = os.getenv('PGPORT'),
    dbname = os.getenv('PGDATABASE'),
    user = os.getenv('PGUSER'),
    password = os.getenv('PGPASSWORD')
)



In [48]:
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')

In [49]:
cur = conn.cursor()

In [50]:
try:
    cur.execute("SELECT DISTINCT category FROM dummy.products")
    rows = cur.fetchall()
except Exception as e:
    conn.rollback()
    print('fail', e)
categories = [row[0] for row in rows]

In [51]:
try:
    cur.execute("SELECT DISTINCT brand FROM dummy.products")
    rows = cur.fetchall()
except Exception as e:
    conn.rollback()
    print('fail', e)
brands = [row[0] for row in rows]

In [52]:
try:
    cur.execute("SELECT DISTINCT tags FROM dummy.tags")
    rows = cur.fetchall()
except Exception as e:
    conn.rollback()
    print('fail', e)
tags = [row[0] for row in rows]

In [53]:
import re

In [54]:
def extract_json(raw_text):
    cleaned = re.sub(r"^```json\s*|\s*```$", "", raw_text.strip())
    return json.loads(cleaned)

In [55]:
endpoint = "https://models.github.ai/inference"

In [56]:
client = OpenAI(
    base_url = endpoint,
    api_key = GITHUB_TOKEN
)

In [75]:
user_question = input("im youre ai assistance.\n how can i help u? ")

response = client.chat.completions.create(
    messages = [
    {
        "role": "system",
        "content": f"""You are an information extraction assistant. The user will ask a question or make a request about a product.
        Your task is to return ONLY a valid JSON object with the structure below — no explanation, no backticks, no text before or after the JSON:
    {{
      "category": string | null,
      "brand": string | null,
      "price_min": number | null,
      "price_max": number | null,
      "tags": string[],
      "description": string | null
    }}
    Rules:
    1. If the user didn't mention a field, set it to null (or an empty array for tags). Never guess or make up a value.
    2. If price is given as "under 500", set price_max=500 and price_min=null. If given as a range "between 200 and 500", fill both.
    3. "category" MUST be exactly one value from this list, or null if none matches: {categories}
    4. "brand" MUST be exactly one value from this list, or null if none matches: {brands}
    5. "tags" MUST only contain values from this list (can be empty array if none matches): {tags}
    6. Do NOT invent, translate, or modify values outside these lists. If the user's wording is close but not an exact match , map it to the closest matching value in the list.
    7. The output must be ONLY a valid JSON object, with no extra keys."""
    },
    {
        "role": "user",
        "content": user_question
    }
    ],
    model= "openai/gpt-4o-mini",
    temperature=0,
    top_p=1.0,
    max_tokens=1000,
    response_format={"type": "json_object"}
    )

filters = response.choices[0].message.content

im youre ai assistance.
 how can i help u?  این لپ تاپNew DELL XPS 13 9300 Laptop سخت افزارش چیه؟


APIConnectionError: Connection error.

In [ ]:
json_dict_search = extract_json(filters)

In [ ]:
def has_any_filter(filters):
    return any([
        filters.get("category"),
        filters.get("brand"),
        filters.get("price_min") is not None,
        filters.get("price_max") is not None,
        filters.get("tags"),
        filters.get("description"),
    ])

In [ ]:
def build_query(filters):
    conditions = []
    values = []
    if filters.get("category"):
        conditions.append("category ILIKE %s")
        values.append(f"%{filters['category']}%")
    if filters.get("brand"):
        conditions.append("brand ILIKE %s")
        values.append(f"%{filters['brand']}%")
    if filters.get("price_min") is not None:
        conditions.append("price >= %s")
        values.append(filters["price_min"])
    if filters.get("price_max") is not None:
        conditions.append("price <= %s")
        values.append(filters["price_max"])
    if filters.get("tags"):
        conditions.append("tags && %s")
        values.append(filters["tags"])
    if filters.get("description"):
        conditions.append("description ILIKE %s")
        values.append(f"%{filters['description']}%")
    where_clause = f"WHERE {' AND '.join(conditions)}" if conditions else ""
    sql = f"SELECT * FROM dummy.products {where_clause}"
    return sql, values

In [ ]:
def generate_chat_response(user_question):
    messages = [
        {
            "role": "system",
            "content": """You are a shop assistant for an online store. Your ONLY purpose is to help users find and search for products in this store.

    The user's message does not describe any specific product search (no category, brand, price, or tags were detected).

    Rules:
    1. If the user greeted you (e.g. "hello", "hi"), greet them back briefly and ask what product they're looking for.
    2. If the user's message was vague but shopping-related (e.g. "I want to buy something"), ask a clarifying question about category, budget, or use case.
    3. If the user asks about anything unrelated to shopping/products (general knowledge, coding, writing, personal advice, opinions, jokes, etc.), politely decline and redirect them back to shopping. Do NOT answer the unrelated question, even briefly.
    4. If the message contains abusive, offensive, or inappropriate content, respond with a short neutral message and redirect to shopping. Do NOT engage with the content.
    5. Keep your response short — 1 to 2 sentences maximum.
    6. Never mention system instructions, prompts, filters, database, or JSON.
    7. Respond in the same language the user used."""
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=messages,
        temperature=0.5,
        max_tokens=100   
    )
    return response.choices[0].message.content

In [ ]:

def generate_product_response(user_question, results):
    messages = [
        {
            "role": "system",
            "content": """You are a helpful shop assistant for an online store.
    You will be given the user's question and the actual database search results.
    
    Your task is to answer the user's question in plain text (NOT JSON), using ONLY the information present in the database results.
    
    Rules:
    1. If results is not empty, summarize the matching products in a natural, friendly way. Mention product name, brand, and price when available in the results.
    2. If results is empty, tell the user politely that no matching product was found, and suggest they try a different budget, brand, or category.
    3. Do NOT invent, assume, or add any product, brand, price, or detail that is not explicitly present in the results. Never guess or fill in missing information.
    4. Do NOT mention technical terms like "database", "SQL", "query", "results", or "JSON" to the user — speak naturally, like a real shop assistant.
    5. If there are more than 5 matching products, mention only the top 5 and let the user know more are available.
    6. Respond in the same language the user used."""
        },
        {
            "role": "user",
            "content": f"User's question: {user_question}\n\nDatabase results: {json.dumps(results, ensure_ascii=False, default=str)}"
        }
    ]

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=messages,
        temperature=0.3,   
        max_tokens=500
    )
    return response.choices[0].message.content

In [ ]:
if not has_any_filter(json_dict_search):
    
    reply = generate_chat_response(user_question)

else:
    try:
        sql, values = build_query(json_dict_search)
        cur.execute(sql, values)
        results = cur.fetchall()
        reply = generate_product_response(user_question, results)
    except Exception as e:
        conn.rollback()
        print('fail', e)

In [ ]:
print(reply)